# Promptfoo 介绍

**注意：这节课位于一个包含相关代码文件的文件夹中。如果你想跟随并自行运行评估，请下载整个文件夹**

我们已经了解了如何从头开始编写自己的评估，虽然有效，但多少有些繁琐。使用专门为此目的设计的工具通常更加实用。如今有多种评估工具和库可供选择（而且还在不断发布！），包括：
- [promptfoo](https://github.com/promptfoo/promptfoo)
- [Vellum](https://www.vellum.ai/#playground)
- [Scale Evaluation](https://scale.com/evaluation/model-developers)
- [Prompt Layer](https://promptlayer.com/)
- [Chain Forge](https://github.com/ianarawjo/ChainForge)
- 还有很多其他工具！

Promptfoo 是一个开源且易于使用的选择。它提供了精简的、开箱即用的解决方案，可以显著减少全面提示词测试所需的时间和精力。它提供了简单、现成的基础设施，用于批量测试、版本控制和性能分析，让开发者能够专注于完善提示词，而不是构建和维护测试框架。它可以轻松跨多个提示词、模型和提供商运行评估，还提供了便于可视化和比较评估结果的工具。Promptfoo 和其他评估工具比从零开始编写自己的评估逻辑要强大得多！

运行评估后，promptfoo 会生成一个仪表板，就像这张图片中所示的那样：



让我们开始吧！

---

## 我们的第一个 promptfoo 评估

本课程的接下来的几节课将专注于使用 promptfoo 编写评估。在第一课中，我们将学习一种简单的方法来使用 promptfoo 评估几节课前那个 "这个动物有多少条腿？" 的提示词。这是一个非常简单的提示词和评估。我们的重点是使用 promptfoo 运行评估的实际工具和流程。

回顾一下，在那节课中我们使用了这个小型评估数据集：

```py
eval_data = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {"animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.", "golden_answer": "5"},
    {"animal_statement": "The animal is a dog.", "golden_answer": "4"},
    {"animal_statement": "The animal is a cat with two extra legs.", "golden_answer": "6"},
    {"animal_statement": "The animal is an elephant.", "golden_answer": "4"},
    {"animal_statement": "The animal is a bird.", "golden_answer": "2"},
    {"animal_statement": "The animal is a fish.", "golden_answer": "0"},
    {"animal_statement": "The animal is a spider with two extra legs", "golden_answer": "10"},
    {"animal_statement": "The animal is an octopus.", "golden_answer": "8"},
    {"animal_statement": "The animal is an octopus that lost two legs and then regrew three legs.", "golden_answer": "9"},
    {"animal_statement": "The animal is a two-headed, eight-legged mythical creature.", "golden_answer": "8"},
]
```

在那节课中，我们编写了三个不同的提示词，它们从我们的粗略评估函数中获得了逐步提高的准确率得分。在本节课中，我们将把评估数据集和提示词移植到 promptfoo 中，看看运行和比较它们的输出是多么容易。

---

## 安装 promptfoo

使用 promptfoo 的第一步是通过命令行安装它。导航到你要编写评估代码的文件夹，然后运行以下命令：

```bash
npx promptfoo@latest init
```

这会在当前目录中创建一个 `promptfooconfig.yaml` 文件。这个文件是所有配置发生的地方。在其中，我们配置以下内容：
- 提供者（我们想在评估中使用的 Anthropic API 模型）
- 我们想要评估的提示词
- 我们想要运行的测试

---



## 配置提供者
接下来，我们可以配置 promptfoo 使用我们想要运行评估的特定 Anthropic API 模型。为此，我们在 `promptfooconfig.yaml` 文件中指定一个 `providers` 字段，并将其设置为一个或多个 Anthropic 模型。Promptfoo 使用特定的模式来指定模型名称。当前支持的 Anthropic 模型字符串有：

- `anthropic:messages:claude-3-5-sonnet-20240620`
- `anthropic:messages:claude-3-haiku-20240307`
- `anthropic:messages:claude-3-sonnet-20240229`
- `anthropic:messages:claude-3-opus-20240229`
- `anthropic:messages:claude-2.0`
- `anthropic:messages:claude-2.1`
- `anthropic:messages:claude-instant-1.2`

在第一次评估中我们将使用 Haiku。删除 `promptfooconfig.yaml` 文件的现有内容，并将其替换为：

```yaml
description: "Animal Legs Eval"
  
providers:
  - "anthropic:messages:claude-3-haiku-20240307"
```

以下是每个部分的详细说明：

- `description` 是一个可选的标签，描述我们正在评估的任务。
- `providers` 告诉 promptfoo 我们想在这个评估中使用 Haiku。我们可以指定多个模型，这在以后的课程中会看到。



运行评估时，promptfoo 会查找 `ANTHROPIC_API_KEY` 环境变量。你可以通过在命令行中运行以下命令来设置环境变量：

```bash
export ANTHROPIC_API_KEY=your_api_key_here
```

---

## 指定我们的提示词
下一步是告诉 promptfoo 我们想要评估的提示词。有很多方法可以做到这一点，包括：
- 将提示词直接放在 YAML 文件中作为文本
- 从 JSON 文件加载提示词
- 从文本文件加载提示词
- 从另一个 YAML 文件加载提示词
- 从 Python 文件加载提示词

我们更喜欢将所有相关提示词放在一个 Python 文件中作为返回提示词字符串的单独函数。在后面的课程中我们将看到其他方法。Promptfoo 非常灵活，你将在本课程中看到！

创建一个名为 `prompts.py` 的 Python 文件，并在其中放入以下提示词函数：

```py
def simple_prompt(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.
    
    Here is the animal statement.
    <animal_statement>{animal_statement}</animal_statement>
    
    How many legs does the animal have? Please respond with a number"""

def better_prompt(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.
    
    Here is the animal statement.
    <animal_statement>{animal_statement}</animal_statement>
    
    How many legs does the animal have? Please only respond with a single digit like 2 or 9"""
```

注意这些函数都接受一个 `animal_statement` 参数，将其插入到提示词中，然后返回最终的提示词字符串。

下一步是告诉 promptfoo 配置文件我们想要从刚创建的 `prompts.py` 文件加载提示词。为此，更新 `promptfooconfig.yaml` 文件以包含此代码：

```yaml
description: "Animal Legs Eval"

prompts:
  - prompts.py:simple_prompt
  - prompts.py:better_prompt
  
providers:
  - "anthropic:messages:claude-3-haiku-20240307"
```

注意我们为 `prompts.py` 文件中的每个提示词函数添加了单独的一行。我们已经告诉 promptfoo 我们希望评估两个提示词：`simple_prompt` 和 `better_prompt`，它们都"位于" `prompts.py` 文件中。

---

## 配置我们的测试

下一步是告诉 promptfoo 我们想要用特定的提示词和提供者运行哪些测试。Promptfoo 为我们提供了许多定义测试的选项，但我们将从最常见的方法开始：在 CSV 文件中指定我们的测试。

我们将创建一个名为 `dataset.csv` 的新 CSV 文件，并将测试输入写入其中。

Promptfoo 允许我们在 CSV 文件中直接定义评估逻辑。在接下来的课程中我们将看到 promptfoo 自带的一些内置测试断言，但对于这个特定的评估，我们只需要检查模型输出与预期输出的腿数之间是否完全匹配。

为此，我们用两个列标题编写 CSV：
- `animal_statement` - 包含输入的动物陈述，如 "The animal is an elephant"
- `__expected` - 包含预期的正确输出（注意 `__expected` 中的双下划线）。这是 promptfoo 特定的语法。


创建一个 `dataset.csv` 文件并添加以下内容：

```csv
animal_statement,__expected
"The animal is a human.","2"
"The animal is a snake.","0"
"The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.","5"
"The animal is a dog.","4"
"The animal is a cat with two extra legs.","6"
"The animal is an elephant.","4"
"The animal is a bird.","2"
"The animal is a fish.","0"
"The animal is a spider with two extra legs","10"
"The animal is an octopus.","8"
"The animal is an octopus that lost two legs and then regrew three legs.","9"
"The animal is a two-headed, eight-legged mythical creature.","8"
```

最后，我们要告诉 promptfoo 它应该使用我们的 `dataset.csv` 文件来加载测试。为此，更新 `promptfooconfig.yaml` 文件以包含此代码：

```yaml
description: "Animal Legs Eval"

prompts:
  - prompts.py:simple_prompt
  - prompts.py:better_prompt
  
providers:
  - "anthropic:messages:claude-3-haiku-20240307"

tests: animal_legs_tests.csv
```

---


## 运行我们的评估

现在我们已经指定了提供者、提示词和测试，是时候运行评估了！

在终端中运行以下命令：

```bash
npx promptfoo@latest eval
```
这将启动评估过程。对于我们的每个提示词，promptfoo 将：
- 从 CSV 文件中获取每个 `animal_statement`
- 构建包含 `animal_statement` 的完整提示词
- 使用单独的提示词向 Anthropic API 发送请求
- 检查输出是否与 CSV 文件中的预期输出匹配

评估完成后，promptfoo 将在终端中显示结果。


这是运行上述代码的示例 promptfoo 输出：



上面的截图只包含前四行，但评估确实在所有十二个输入上运行了。
- 左侧列显示特定的 `animal_statement`
- 中间列显示 `simple_prompt` 的输出和分数，这个提示词似乎在每个测试用例上都失败了！
- 右侧列显示 `better_prompt` 的输出和分数，除了逻辑复杂的测试用例外，它在大多数测试用例上都成功了。

---

## 查看评估结果

Promptfoo 使启动仪表板变得非常容易，可以在浏览器中可视化和检查评估结果。运行上述评估后，尝试在终端中运行此命令：

```bash
npx promptfoo@latest view
```

这会询问你是否要启动服务器（输入 'y'），然后在浏览器中打开仪表板。



最相关的摘要信息在顶部：



我们还可以深入查看特定结果，了解它们失败的原因。让我们看一下 `simple_prompt` 结果之一（中间列）。这个提示词的每一行都标记为失败。怎么回事？

点击单元格中的放大镜按钮了解更多：



这会打开一个包含输出和评分详细信息的模态框：



我们可以清楚地看到，这个 `simple_prompt` 得到了正确的答案 0，但输出包含大量额外的解释性文本，导致评估失败。

如果我们仔细观察最右边的列，其中包含我们 `better_prompt` 提示词的结果，我们会得到更好的响应，都是像 `5` 或 `0` 这样的个位数字。它似乎在更复杂的 `animal_statements` 上失败了，这些需要更多推理才能回答，例如：

> The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.

---


## 添加第三个提示词
回想一下之前关于代码分级评估的课程，我们通过在提示词中添加一些思维链推理获得了最佳结果。让我们添加一个包含思维链的改进的第三个提示词，看看它对"棘手"问题的表现如何！

将以下提示词函数添加到 `prompts.py`：

```py
def chain_of_thought_prompt(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.
    
    Here is the animal statement.
    <animal_statement>{animal_statement}</animal_statement>
    
    How many legs does the animal have? 
    Start by reasoning about the numbers of legs the animal has, thinking step by step inside of <thinking> tags.  
    Then, output your final answer inside of <answer> tags. 
    Inside the <answer> tags return just the number of legs as an integer and nothing else."""
```

接下来，更新 `promptfooconfig.yaml` 文件以包含我们的新提示词：


```yaml
description: "Animal Legs Eval"

prompts:
  - prompts.py:simple_prompt
  - prompts.py:better_prompt
  - prompts.py:chain_of_thought_prompt
  
providers:
  - "anthropic:messages:claude-3-haiku-20240307"

tests: animal_legs_tests.csv
```


在我们运行此评估之前，我们必须解决一个问题：我们新的 `chain_of_thought_prompt` 在输出中包含 `<thinking>` 和 `<answer>` 标签。为了真正评估模型使用此提示词的表现，我们需要提取模型放在 `<answer>` 标签内的数字答案，并将其与预期值进行比较。

Promptfoo 允许我们定义自定义的 `transforms`，我们可以用它来在执行实际比较逻辑之前操作模型的输出。为此，我们将编写一个简单的 Python 函数来从 `<answer>` 标签之间提取答案。

创建一个名为 `transform.py` 的新文件，并向其中添加以下代码：


```py
def get_transform(output, context):
    if "<thinking>" in output:
        try:
            return output.split("<answer>")[1].split("</answer>")[0].strip()
        except Exception as e:
            print(f"Error in get_transform: {e}")
            return output
    return output
```


这个名为 `get_transform` 的函数期望接收模型的输出（我们将在以后的课程中介绍 `context` 参数）。然后我们可以将模型的输出转换为我们想要的任何形式再返回。在这种情况下，我们做两件事之一：

- 如果输出包含 `<thinking>` 标签，我们知道它来自我们的思维链提示词。我们从 `<answer>` 标签之间提取数字并将其作为新输出返回。
- 否则只返回原始输出（对于其他不使用思维链的提示词）

最后一步是告诉 promptfoo 我们想要使用这个转换函数。更新 `promptfooconfig.yaml` 文件如下：

```yaml
description: "Animal Legs Eval"

prompts:
  - prompts.py:simple_prompt
  - prompts.py:better_prompt
  - prompts.py:chain_of_thought_prompt
  
providers:
  - "anthropic:messages:claude-3-haiku-20240307"

tests: animal_legs_tests.csv

defaultTest:
  options:
    transform: file://transform.py
```

最后一部分告诉 promptfoo 始终对我们的所有测试应用 `transform.py` 中的转换函数。默认情况下，promptfoo 会在 `transform.py` 文件中查找名为 `get_transform` 的函数。

现在我们可以使用以下命令再次运行评估：

```bash
npx promptfoo@latest eval
```

我们会看到类似这样的输出，现在包含 4 列：



我们可以使用以下命令再次在浏览器中查看结果：

```bash
npx promptfoo@latest view
```
我们会看到这样的网页：



我们可以清楚地看到，包含思维链的提示词在所有问题上都得到了 100% 的正确率！

---

## 比较模型
Promptfoo 的一个很好的特性是使用不同模型运行评估非常容易。我们必须进行一些提示词工程工作才能在使用 Haiku 时获得 100% 的提示词得分，但让我们看看如果切换到像 Claude 3.5 Sonnet 这样更有能力的模型会发生什么。

我们只需要更新 `promptfooconfig.yaml` 文件，添加一个与有效的 Anthropic 提供者字符串匹配的第二个提供者。更新 `promptfooconfig.yaml` 以包含两个提供者：


```yaml
description: "Animal Legs Eval"

prompts:
  - prompts.py:simple_prompt
  - prompts.py:better_prompt
  - prompts.py:chain_of_thought_prompt
  
providers:
  - anthropic:messages:claude-3-haiku-20240307
  - anthropic:messages:claude-3-5-sonnet-20240620

tests: animal_legs_tests.csv

defaultTest:
  options:
    transform: file://transform.py
```

然后我们可以使用与之前相同的命令再次运行评估：

```bash
npx promptfoo@latest eval
```

当我们查看基于网络的仪表板时，会看到一些有趣的结果！



只需在 YAML 文件中添加一行，我们就能够在两个模型上运行评估集。前三列输出来自 Claude 3 Haiku，最后三列输出来自 Claude 3.5 Sonnet。看起来 Claude 3.5 Sonnet 即使使用在 Claude 3 Haiku 上得分为 0% 的 `simple_prompt` 也能以 100% 通过我们的评估。

这类信息非常有价值：不仅仅是哪个提示词表现最好，而是对于给定任务，哪种模型+提示词组合表现最好。

**旁注：** 如果你好奇为什么 Claude 3.5 Sonnet 在思维链提示词上没有达到 100%，这里有解释！它在 `animal_statement` "The animal is an octopus." 的测试中出错。在其 `<thinking>` 标签内，Claude 3.5 Sonnet 推理说章鱼实际上没有任何腿，而是有通常被称为"手臂"但从不是"腿"的附肢。通过升级到"更智能"的模型，我们实际上看到思维链提示词的表现略差，因为模型变得"太智能"了。如果我们要确保在所有模型上的表现，我们可以更新提示词以更具体地说明什么才算"腿"。

这只是我们对 promptfoo 的初步体验。在未来的课程中，我们将学习如何处理更复杂的代码分级逻辑、定义我们自己的自定义评分器，以及运行模型分级评估。